In [1]:
# 1. Create Project Directory Structure (local-inside Colab)
import os

base_dir = "/content/wildfire-prediction"
os.makedirs(base_dir, exist_ok=True)

for d in ["data", "results", "models"]:
    os.makedirs(os.path.join(base_dir, d), exist_ok=True)

os.listdir(base_dir)

['results', 'models', 'data']

In [2]:
# 2. Import Dataset from Kaggle
import kagglehub

# Download the dataset from Kaggle
path = kagglehub.dataset_download("abdelghaniaaba/wildfire-prediction-dataset")
print("Dataset downloaded to:", path)


100%|██████████| 1.45G/1.45G [01:09<00:00, 22.4MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/abdelghaniaaba/wildfire-prediction-dataset/versions/1


In [3]:
# 3. Copy Dataset into the local data folder (train/valid/test)
import shutil

src = path
dst = os.path.join(base_dir, "data")

for split in ["train", "valid", "test"]:
    src_split = os.path.join(src, split)
    dst_split = os.path.join(dst, split)
    if os.path.exists(src_split):
        shutil.copytree(src_split, dst_split, dirs_exist_ok=True)
        print(f"Copied {split} folder successfully.")
    else:
        print(f"Folder {split} not found in source path.")

print("Dataset copied to data/")


Copied train folder successfully.
Copied valid folder successfully.
Copied test folder successfully.
Dataset copied to data/


In [4]:
# 4. Verify Folder Structure and Class Counts
for split in ["train", "valid", "test"]:
    split_dir = os.path.join(dst, split)
    print(f"\n[{split.upper()}]")
    if not os.path.exists(split_dir):
        print("  Missing folder.")
        continue
    for cls in os.listdir(split_dir):
        cls_path = os.path.join(split_dir, cls)
        if os.path.isdir(cls_path):
            num_files = sum(len(files) for _, _, files in os.walk(cls_path))
            print(f"  Class: {cls:15s} -> {num_files} images")



[TRAIN]
  Class: wildfire        -> 15750 images
  Class: nowildfire      -> 14500 images

[VALID]
  Class: wildfire        -> 3480 images
  Class: nowildfire      -> 2820 images

[TEST]
  Class: wildfire        -> 3480 images
  Class: nowildfire      -> 2820 images


In [5]:
# 5. Define Preprocessing & Augmentation Pipelines
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Define image transformations
train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

valid_test_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Create datasets
train_dataset = datasets.ImageFolder(root=os.path.join(dst, "train"), transform=train_transform)
valid_dataset = datasets.ImageFolder(root=os.path.join(dst, "valid"), transform=valid_test_transform)
test_dataset  = datasets.ImageFolder(root=os.path.join(dst, "test"),  transform=valid_test_transform)

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

# Summary
class_names = train_dataset.classes
print("Classes:", class_names)
print("Train size:", len(train_dataset))
print("Valid size:", len(valid_dataset))
print("Test size:", len(test_dataset))


Classes: ['nowildfire', 'wildfire']
Train size: 30250
Valid size: 6300
Test size: 6300


In [7]:
# 6. Compute Class Weights (to handle imbalance)
import numpy as np

labels = np.array([y for _, y in train_dataset.samples])
class_sample_counts = np.bincount(labels).astype(int)

print("Train class counts:", {cls: int(n) for cls, n in zip(class_names, class_sample_counts)})

# Compute normalized inverse-frequency weights
class_weights = 1.0 / (class_sample_counts + 1e-6)
class_weights = class_weights / class_weights.sum() * len(class_weights)
class_weights = class_weights.astype(float)

print("Class weights:", {cls: round(float(w), 4) for cls, w in zip(class_names, class_weights)})

# Save weights for training
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)


Train class counts: {'nowildfire': 14500, 'wildfire': 15750}
Class weights: {'nowildfire': 1.0413, 'wildfire': 0.9587}
